# 03 — ASR: Wav2Vec2 Fine-tuning (Kaggle / Colab / Local)

Fine-tunes `nguyenvulebinh/wav2vec2-base-vi-vlsp2020` trên ViMD.

## ✅ Fixes applied

| # | Nguồn | Bug | Fix |
|---|-------|-----|-----|
| 1 | fix.md | AMP + CTC → NaN/inf | `USE_AMP=False` |
| 2 | fix.md | LR=1e-4 quá cao | `LR=1e-5` |
| 3 | fix.md | `scheduler.step()` sai vị trí | Step sau mỗi optimizer.step() |
| 4 | fix.md | audio trim → sample rỗng | zeros fallback |
| 5 | fix.md | Label không giới hạn độ dài | `MAX_LABEL_LEN=256` |
| 6 | fix.md | Không chặn NaN trước backward | Skip batch nếu NaN/inf |
| 7 | trước | `freeze_feature_extractor()` deprecated | `requires_grad_(False)` |
| 8 | trước | Thiếu `attention_mask` | `return_attention_mask=True` |
| 9 | trước | `skip_special_tokens` thiếu | Fixed |

**Config**: 3 epochs test · Kaggle T4/P100 · `USE_AMP=False`

In [ ]:
# ── Cell 0: Environment + Paths ───────────────────────────────────────────────
import sys, os

if os.path.exists('/kaggle/working/project/training/notebooks'):
    sys.path.insert(0, '/kaggle/working/project/training/notebooks')

from _env import *
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ENV          : {ENV_NAME}')
print(f'Device       : {device}')
if device == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  Không có GPU!')
print(f'DATASET_PATH : {FINAL_DATASET}')
print(f'CHECKPOINT   : {ASR_CKPT_DIR}')

In [ ]:
# ── Cell 1: Dependencies ──────────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.40.0', 'datasets>=2.18.0', 'accelerate>=0.28.0',
                'evaluate>=0.4.0', 'jiwer>=3.0.0', 'librosa', 'soundfile'], check=False)
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Load Dataset ──────────────────────────────────────────────────────
from datasets import load_from_disk

if not os.path.exists(FINAL_DATASET):
    raise RuntimeError(f'Dataset không tìm thấy: {FINAL_DATASET}\nChạy 01_dataset_prep.ipynb trước.')

ds = load_from_disk(FINAL_DATASET)
print('Dataset:', ds)
n_audio = len(os.listdir(AUDIO_DIR)) if os.path.isdir(AUDIO_DIR) else 0
print(f'AUDIO_DIR: {n_audio:,} files')
if n_audio == 0:
    print('⚠️  Audio dir rỗng — chạy lại 01_dataset_prep Cell 5')

In [ ]:
# ── Cell 3: Load Model + Processor ────────────────────────────────────────────
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_NAME = 'nguyenvulebinh/wav2vec2-base-vi-vlsp2020'
processor  = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model      = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction='mean',
    pad_token_id=processor.tokenizer.pad_token_id,
).to(device)

# ✅ Fix 7: freeze_feature_extractor() deprecated >= 4.37
try:
    model.freeze_feature_extractor()
    print('freeze_feature_extractor() OK')
except AttributeError:
    model.wav2vec2.feature_extractor.requires_grad_(False)
    print('✓ Feature extractor frozen (requires_grad_(False))')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.1f}%)')

In [ ]:
# ── Cell 4: Tokenizer Diagnostic ──────────────────────────────────────────────
# ✅ Fix 9: kiểm tra UNK trước khi train
TEXT_COL = 'text' if 'text' in ds['train'].column_names else 'transcript'
print(f'TEXT_COL: {TEXT_COL}')
print(f'Vocab: {len(processor.tokenizer)} | pad_id: {processor.tokenizer.pad_token_id}')

unk_found = False
for t in ds['train'][:5][TEXT_COL]:
    if not t or not str(t).strip(): continue
    t = str(t).strip().lower()
    ids = processor.tokenizer(t).input_ids
    unk = processor.tokenizer.unk_token_id
    has_unk = unk is not None and unk in ids
    if has_unk: unk_found = True
    print(f'  {t[:50]} → {"❌UNK" if has_unk else "✓"}')

if unk_found:
    print('\n❌ UNK tokens phát hiện — cần kiểm tra text normalization')
else:
    print('\n✓ Tokenizer OK')

In [ ]:
# ── Cell 5: DataLoader + Collate ──────────────────────────────────────────────
import numpy as np, librosa, soundfile as sf, io
from torch.utils.data import DataLoader

MAX_SAMPLES   = int(10.0 * 16000)  # 10s
MAX_LABEL_LEN = 256                # ✅ Fix 5: giới hạn label length

def read_audio(fn: str) -> np.ndarray:
    with open(os.path.join(AUDIO_DIR, fn), 'rb') as f:
        b = f.read()
    data, sr = sf.read(io.BytesIO(b), dtype='float32')
    if data.ndim > 1: data = data.mean(axis=1)
    if sr != 16000: data = librosa.resample(data, orig_sr=sr, target_sr=16000)
    # ✅ Fix 4: audio trim rỗng → zeros
    data, _ = librosa.effects.trim(data, top_db=20)
    if len(data) == 0:
        data = np.zeros(1600, dtype=np.float32)
    return data[:MAX_SAMPLES].astype(np.float32)

def collate_fn(batch):
    audios, labels = [], []
    for item in batch:
        try: audios.append(read_audio(item['filename']))
        except Exception: audios.append(np.zeros(1600, dtype=np.float32))
        t   = str(item.get(TEXT_COL, item.get('text', ''))).strip().lower()
        ids = processor.tokenizer(t).input_ids[:MAX_LABEL_LEN] if t else []
        labels.append(ids if ids else [processor.tokenizer.pad_token_id])

    # ✅ Fix 8: return_attention_mask=True
    inputs = processor(audios, sampling_rate=16000, return_tensors='pt',
                       padding=True, return_attention_mask=True)
    max_ll = max(len(l) for l in labels)
    inputs['labels'] = torch.tensor(
        [l + [-100] * (max_ll - len(l)) for l in labels], dtype=torch.long)
    return inputs

def has_audio(item):
    return os.path.exists(os.path.join(AUDIO_DIR, item['filename']))

ds_train = ds['train'].filter(has_audio, desc='Filter train')
ds_valid = ds['valid'].filter(has_audio, desc='Filter valid')
ds_test  = ds['test'].filter(has_audio,  desc='Filter test')
print(f'Train: {len(ds_train):,} | Valid: {len(ds_valid):,} | Test: {len(ds_test):,}')

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(ds_valid, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS)
print(f'Batch_size={BATCH_SIZE} | Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}')

# Quick verify
s = next(iter(DataLoader(ds_train.select(range(min(2,len(ds_train)))),
                          batch_size=2, collate_fn=collate_fn)))
assert 'attention_mask' in s, '❌ attention_mask MISSING!'
print(f'✓ attention_mask shape: {s["attention_mask"].shape}')

In [ ]:
# ── Cell 6: Training Loop ─────────────────────────────────────────────────────
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import evaluate, json

wer_metric = evaluate.load('wer')

# ─── Hyperparameters ──────────────────────────────────────────────────────────
NUM_EPOCHS    = 3            # test run — tăng lên 15 sau khi verify WER giảm
# ✅ Fix 2: LR=1e-5 (fix.md: 1e-4 quá cao cho wav2vec2-base full finetune)
LR            = 1e-5
GRAD_ACC      = 8            # effective batch = BATCH_SIZE × GRAD_ACC
MAX_GRAD_NORM = 1.0
WARMUP_STEPS  = 100
# ✅ Fix 1: AMP=False vì fp16 + CTC sinh inf → nan
USE_AMP       = False

# ─── Resume từ checkpoint ─────────────────────────────────────────────────────
CKPT_STATE = os.path.join(ASR_CKPT_DIR, 'train_state.json')
start_epoch = 0
best_wer    = float('inf')

if os.path.exists(CKPT_STATE):
    with open(CKPT_STATE) as f:
        ckpt = json.load(f)
    start_epoch = ckpt.get('epoch', 0)
    best_wer    = ckpt.get('best_wer', float('inf'))
    model_ckpt  = os.path.join(ASR_CKPT_DIR, 'config.json')
    if os.path.exists(model_ckpt):
        model = Wav2Vec2ForCTC.from_pretrained(ASR_CKPT_DIR).to(device)
        try: model.freeze_feature_extractor()
        except AttributeError: model.wav2vec2.feature_extractor.requires_grad_(False)
        print(f'✅ Resume: epoch={start_epoch}, best_wer={best_wer:.2f}%')
    else:
        start_epoch = 0

if start_epoch >= NUM_EPOCHS:
    print(f'✅ Training done ({start_epoch} epochs). Chạy Cell 8.')
else:
    optimizer   = AdamW([p for p in model.parameters() if p.requires_grad],
                         lr=LR, weight_decay=0.01, betas=(0.9, 0.98))
    total_steps = NUM_EPOCHS * max(1, len(train_loader) // GRAD_ACC)
    # ✅ Fix 3: LinearScheduler với warmup
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS,
        num_training_steps=max(total_steps, 1))

    if start_epoch > 0:
        for _ in range(start_epoch * max(1, len(train_loader) // GRAD_ACC)):
            scheduler.step()

    # ✅ Fix 1: scaler chỉ dùng khi USE_AMP=True
    scaler = torch.cuda.amp.GradScaler() if (USE_AMP and device == 'cuda') else None

    print(f'Config: epochs={NUM_EPOCHS} | LR={LR} | GRAD_ACC={GRAD_ACC}')
    print(f'        USE_AMP={USE_AMP} | batch={BATCH_SIZE} | warmup={WARMUP_STEPS}')
    print(f'        total_steps={total_steps} | resume_from={start_epoch}\n')

    global_step = start_epoch * max(1, len(train_loader) // GRAD_ACC)
    nan_count   = 0

    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        total_loss = 0.0
        valid_steps = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', dynamic_ncols=True)
        for step, batch in enumerate(pbar):
            iv   = batch['input_values'].to(device)
            mask = batch['attention_mask'].to(device)   # ✅ Fix 8
            lbl  = batch['labels'].to(device)

            # ✅ Fix 1: USE_AMP=False → không dùng autocast
            if scaler and USE_AMP:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(input_values=iv, attention_mask=mask, labels=lbl)
                loss = outputs.loss / GRAD_ACC
                # ✅ Fix 6: NaN check trước khi backward
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_count += 1; optimizer.zero_grad(); continue
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACC == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    # ✅ Fix 3: scheduler.step() sau optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1
            else:
                outputs = model(input_values=iv, attention_mask=mask, labels=lbl)
                loss    = outputs.loss / GRAD_ACC
                # ✅ Fix 6: NaN check
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_count += 1; optimizer.zero_grad(); continue
                loss.backward()
                if (step + 1) % GRAD_ACC == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    optimizer.step()
                    # ✅ Fix 3: step sau optimizer
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

            total_loss  += loss.item() * GRAD_ACC
            valid_steps += 1
            pbar.set_postfix(loss=f'{loss.item()*GRAD_ACC:.4f}',
                             lr=f'{scheduler.get_last_lr()[0]:.2e}', nan=nan_count)

        avg_loss = total_loss / valid_steps if valid_steps > 0 else float('nan')

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        all_preds, all_refs = [], []
        blank_total = pred_total = 0
        first_debug = True

        with torch.no_grad():
            for vb in tqdm(val_loader, desc='Val', leave=False):
                logits   = model(input_values=vb['input_values'].to(device),
                                 attention_mask=vb['attention_mask'].to(device)).logits
                pred_ids = logits.argmax(dim=-1)
                preds    = processor.batch_decode(pred_ids)
                vlbl     = vb['labels'].clone()
                vlbl[vlbl == -100] = processor.tokenizer.pad_token_id
                # ✅ Fix 9: skip_special_tokens
                refs     = processor.tokenizer.batch_decode(vlbl, skip_special_tokens=True)
                all_preds.extend(preds); all_refs.extend(refs)
                blank_total += (pred_ids == processor.tokenizer.pad_token_id).sum().item()
                pred_total  += pred_ids.numel()
                if first_debug:
                    print(f'\n[Val Debug] Epoch {epoch+1}:')
                    for p, r in zip(preds[:3], refs[:3]):
                        print(f"  PRED: '{p[:60]}'")
                        print(f"  REF : '{r[:60]}'")
                    first_debug = False

        blank_ratio = blank_total / pred_total if pred_total > 0 else 1.0
        pairs = [(p, r) for p, r in zip(all_preds, all_refs) if r.strip()]
        wer   = 100 * wer_metric.compute(
            predictions=[p for p,r in pairs], references=[r for p,r in pairs]
        ) if pairs else 100.0

        print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}: '
              f'loss={avg_loss:.4f} | WER={wer:.2f}% | '
              f'blank={blank_ratio:.1%} | NaN={nan_count}')

        if blank_ratio > 0.95:
            print('  ℹ️  Blank>95% — bình thường epoch 1-2 CTC')

        # Save checkpoint (Kaggle: persist khi Save Version)
        if wer < best_wer:
            best_wer = wer
            model.save_pretrained(ASR_CKPT_DIR)
            processor.save_pretrained(ASR_CKPT_DIR)
            print(f'  ✅ Best checkpoint saved (WER={wer:.2f}%)')

        with open(CKPT_STATE, 'w') as f:
            json.dump({'epoch': epoch+1, 'best_wer': best_wer,
                       'global_step': global_step, 'nan_count': nan_count}, f, indent=2)
        print(f'  💾 State saved → {CKPT_STATE}')

    print(f'\n✅ Training done. Best WER: {best_wer:.2f}%')
    if IS_KAGGLE:
        print('\n⚠️  Kaggle: Nhấn "Save Version" để persist checkpoint!')

In [ ]:
# ── Cell 7: Export best model ─────────────────────────────────────────────────
import shutil
if os.path.exists(os.path.join(ASR_CKPT_DIR, 'config.json')):
    os.makedirs(ASR_WEIGHT_DIR, exist_ok=True)
    for fn in os.listdir(ASR_CKPT_DIR):
        src = os.path.join(ASR_CKPT_DIR, fn)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(ASR_WEIGHT_DIR, fn))
    print(f'✅ Exported → {ASR_WEIGHT_DIR}')
else:
    print('⚠️  Chưa có checkpoint')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhấn "Save Version" để lưu model!')

In [ ]:
# ── Cell 8: Test Set Evaluation ───────────────────────────────────────────────
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

eval_src = ASR_WEIGHT_DIR if os.path.exists(os.path.join(ASR_WEIGHT_DIR, 'config.json')) else ASR_CKPT_DIR
eval_model = Wav2Vec2ForCTC.from_pretrained(eval_src).to(device)
eval_proc  = Wav2Vec2Processor.from_pretrained(eval_src)
eval_model.eval()

wer2 = evaluate.load('wer')
all_preds, all_refs = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test'):
        logits   = eval_model(input_values=batch['input_values'].to(device),
                              attention_mask=batch['attention_mask'].to(device)).logits
        pred_ids = logits.argmax(dim=-1)
        preds    = eval_proc.batch_decode(pred_ids)
        lbl      = batch['labels'].clone()
        lbl[lbl == -100] = eval_proc.tokenizer.pad_token_id
        refs     = eval_proc.tokenizer.batch_decode(lbl, skip_special_tokens=True)
        all_preds.extend(preds); all_refs.extend(refs)

pairs = [(p, r) for p, r in zip(all_preds, all_refs) if r.strip()]
test_wer = 100 * wer2.compute(
    predictions=[p for p,r in pairs], references=[r for p,r in pairs]
) if pairs else 100.0

print(f'\n===== Test WER: {test_wer:.2f}% =====')
print(f'Target: ≤ 8.0% | Status: {"✅ PASS" if test_wer <= 8.0 else "📈 In progress"}')
for p, r in list(zip(all_preds[:4], all_refs[:4])):
    print(f'  PRED: {repr(p[:60])}')
    print(f'  REF : {repr(r[:60])}')

## Kết quả kỳ vọng

| Epoch | WER (kỳ vọng) |
|-------|---------------|
| 1 | ~100% (CTC học blank) |
| 2 | 60-100% |
| 3 | 30-80% (fix hoạt động) |
| 10+ | < 20% |

**Nếu WER epoch 3 bắt đầu giảm** → tăng `NUM_EPOCHS = 15` và re-run Cell 6.

**Kaggle**: `Save Version` sau mỗi epoch hoàn tất để không mất checkpoint.